# Scoping Agentic Implementations: Workshop Notebook

This notebook is your working tool for the scoping workshop. It guides you through:
1. Formalizing personas and question taxonomy
2. Generating a seed evaluation dataset from brainstormed questions
3. Using observability data to compare expected vs. actual question distributions
4. Producing a completed agent spec document

**Prerequisites:** Run `setup.sql` first to create the lab database and mock observability data.

In [ ]:
# Connection setup — works in both Snowsight notebooks and local Jupyter
import os

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
        "warehouse": "SCOPING_LAB_WH",
        "database": "SCOPING_LAB",
        "schema": "PUBLIC",
    }
    session = Session.builder.configs(connection_params).create()

session.sql("USE DATABASE SCOPING_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE SCOPING_LAB_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

---
## Activity 1: Persona Mapping

Fill in the persona cards below. Replace the example content with your own use case.
Pick **one persona** as your Phase 1 target.

### Persona 1 (Phase 1 Target)

| Attribute | Your Input |
|-----------|------------|
| **Role** | _e.g., Regional Sales Manager_ |
| **Responsibility** | _e.g., Territory allocation, quota decisions_ |
| **Data Literacy** | _e.g., Reads dashboards, doesn't write SQL_ |
| **Current Workflow** | _e.g., Asks analyst team, 2-day turnaround_ |
| **Frequency** | _e.g., 5-10 questions/week_ |
| **Stakes** | _e.g., Quota misallocation costs $500K/quarter_ |
| **Success Signal** | _e.g., "I got the answer without filing a ticket"_ |

### Persona 2 (Phase 2+)

| Attribute | Your Input |
|-----------|------------|
| **Role** | _e.g., VP of Marketing_ |
| **Responsibility** | _e.g., Budget allocation, board reporting_ |
| **Data Literacy** | _e.g., Needs narrative, not raw numbers_ |
| **Current Workflow** | _e.g., Weekly meeting with analytics lead_ |
| **Frequency** | _e.g., 2-3 questions/week, spikes at quarter-end_ |
| **Stakes** | _e.g., Wrong budget decision = $2M misallocation_ |
| **Success Signal** | _e.g., "I can prep for board meetings without my analyst"_ |

---
## Activity 2: Question Taxonomy

Brainstorm 15-20 questions your Phase 1 persona would actually ask.
Categorize each question and note the data source.

In [ ]:
# Define your question taxonomy as a structured list.
# Replace these examples with questions for YOUR use case.

from collections import Counter

questions = [
    # LOOKUP questions (simple fact retrieval)
    {"question": "What was Q2 revenue for the West region?", "category": "lookup", "data_source": "structured", "risk": "medium"},
    {"question": "How many deals closed last month?", "category": "lookup", "data_source": "structured", "risk": "medium"},
    {"question": "Who is our largest customer by ARR?", "category": "lookup", "data_source": "structured", "risk": "low"},
    
    # AGGREGATION questions (comparisons, trends)
    {"question": "Compare YoY growth by product line", "category": "aggregation", "data_source": "structured", "risk": "high"},
    {"question": "Which region has the highest win rate this quarter?", "category": "aggregation", "data_source": "structured", "risk": "high"},
    {"question": "Show pipeline conversion rates by stage for the last 6 months", "category": "aggregation", "data_source": "structured", "risk": "medium"},
    
    # REASONING questions (multi-step, require interpretation)
    {"question": "Why did the Northeast region underperform last quarter?", "category": "reasoning", "data_source": "structured+unstructured", "risk": "high"},
    {"question": "What's driving the increase in deal cycle time?", "category": "reasoning", "data_source": "structured+unstructured", "risk": "high"},
    
    # POLICY questions (document retrieval)
    {"question": "What's our discount approval process for enterprise deals?", "category": "policy", "data_source": "unstructured", "risk": "very_high"},
    {"question": "What are the territory assignment criteria?", "category": "policy", "data_source": "unstructured", "risk": "high"},
    
    # OUT OF SCOPE (agent should refuse)
    {"question": "Write a cold outreach email for this prospect", "category": "out_of_scope", "data_source": "n/a", "risk": "reputational"},
    {"question": "What's our competitor's pricing?", "category": "out_of_scope", "data_source": "n/a", "risk": "reputational"},
]

print(f"Total questions: {len(questions)}")
print(f"\nBreakdown by category:")
for cat, count in Counter(q['category'] for q in questions).items():
    print(f"  {cat}: {count}")

---
## Activity 3: Tool Selection

Based on your question taxonomy, map categories to Snowflake tools.
Check prerequisites for each tool.

In [ ]:
# Tool mapping — adjust based on your question taxonomy
tool_mapping = {
    "lookup": {
        "tool": "Semantic View + Cortex Analyst",
        "prerequisite": "Well-described semantic view over sales tables",
        "prerequisite_met": False,
        "phase": 1,
    },
    "aggregation": {
        "tool": "Semantic View + Cortex Analyst",
        "prerequisite": "Semantic view with time-series metrics and verified queries",
        "prerequisite_met": False,
        "phase": 1,
    },
    "reasoning": {
        "tool": "Multi-tool Agent (Analyst + Search)",
        "prerequisite": "Both structured data and relevant documents indexed",
        "prerequisite_met": False,
        "phase": 2,
    },
    "policy": {
        "tool": "Cortex Search",
        "prerequisite": "Policy documents chunked and indexed in search service",
        "prerequisite_met": False,
        "phase": 2,
    },
    "out_of_scope": {
        "tool": "Agent Instructions (refusal boundary)",
        "prerequisite": "Clear out-of-scope definition in system prompt",
        "prerequisite_met": True,
        "phase": 1,
    },
}

print("Tool Selection Summary:")
print("=" * 70)
for category, mapping in tool_mapping.items():
    status = "READY" if mapping['prerequisite_met'] else "BLOCKED"
    print(f"\n  [Phase {mapping['phase']}] {category.upper()}")
    print(f"      Tool: {mapping['tool']}")
    print(f"      Prerequisite: {mapping['prerequisite']}")
    print(f"      Status: {status}")

blocked = [k for k, v in tool_mapping.items() if not v['prerequisite_met'] and v['phase'] == 1]
if blocked:
    print(f"\nPhase 1 BLOCKERS: {blocked}")
    print("  Address these prerequisites before building the agent.")

---
## Activity 4: Seed Eval Dataset Generation

Select questions from your taxonomy and generate a seed evaluation dataset.
We'll use `CORTEX.COMPLETE` to draft ground-truth answers, then you validate.

In [ ]:
# Select your seed eval questions (aim for 10 minimum)
# Include: 6 happy-path, 2 edge-case, 2 out-of-scope

seed_eval_questions = [
    {
        "question": "What was Q2 revenue for the West region?",
        "category": "lookup",
        "ground_truth": "Should return approximately $4.2M from the SALES_REVENUE table, filtered to fiscal Q2 2024 and West region.",
    },
    {
        "question": "How many deals closed last month?",
        "category": "lookup",
        "ground_truth": "Should return the count of opportunities with stage='Closed Won' and close_date in the previous calendar month.",
    },
    {
        "question": "Compare YoY growth by product line",
        "category": "aggregation",
        "ground_truth": "Should show revenue for each product line in current year vs prior year, with percentage growth. All product lines represented.",
    },
    {
        "question": "Which region has the highest win rate this quarter?",
        "category": "aggregation",
        "ground_truth": "Should calculate win_rate = closed_won / (closed_won + closed_lost) per region for the current fiscal quarter.",
    },
    {
        "question": "Show pipeline conversion rates by stage for the last 6 months",
        "category": "aggregation",
        "ground_truth": "Should show percentage of opportunities advancing from each stage to the next, over trailing 6 months.",
    },
    {
        "question": "Who is our largest customer by ARR?",
        "category": "lookup",
        "ground_truth": "Should return customer with highest active annual recurring revenue using current contract values.",
    },
    {
        "question": "What was revenue for Q2 2019?",
        "category": "edge_case",
        "ground_truth": "Should state data is not available for that period OR return the value if it exists. Must NOT hallucinate.",
    },
    {
        "question": "Compare revenue for the Midwest region",
        "category": "edge_case",
        "ground_truth": "Should ask for clarification (compare against what?) or note that Midwest may not exist as a region.",
    },
    {
        "question": "Write a cold outreach email for this prospect",
        "category": "out_of_scope",
        "ground_truth": "Should politely refuse. Must NOT generate email content. Should explain scope boundaries.",
    },
    {
        "question": "What's our competitor's pricing?",
        "category": "out_of_scope",
        "ground_truth": "Should refuse. Must NOT speculate. Should explain it only has access to internal data.",
    },
]

print(f"Seed eval dataset: {len(seed_eval_questions)} questions")
print(f"Coverage: {Counter(q['category'] for q in seed_eval_questions)}")

In [ ]:
# Load the seed eval dataset into Snowflake

session.sql("""
    CREATE TABLE IF NOT EXISTS SEED_EVAL_DATASET (
        question VARCHAR,
        category VARCHAR,
        ground_truth VARCHAR
    )
""").collect()

session.sql("TRUNCATE TABLE IF EXISTS SEED_EVAL_DATASET").collect()

for q in seed_eval_questions:
    session.sql(f"""
        INSERT INTO SEED_EVAL_DATASET (question, category, ground_truth)
        VALUES (
            $${q['question']}$$,
            $${q['category']}$$,
            $${q['ground_truth']}$$
        )
    """).collect()

print(f"Loaded {len(seed_eval_questions)} questions into SEED_EVAL_DATASET")
session.sql("SELECT category, COUNT(*) as cnt FROM SEED_EVAL_DATASET GROUP BY category ORDER BY cnt DESC").show()

In [ ]:
# Use CORTEX.COMPLETE to validate ground-truth quality

sample_question = seed_eval_questions[0]

prompt = f"""You are validating ground-truth descriptions for an AI agent evaluation dataset.

Question: {sample_question['question']}
Category: {sample_question['category']}
Current ground truth: {sample_question['ground_truth']}

Assess this ground truth description. Is it:
1. Specific enough to validate an agent's answer? (not too vague)
2. Flexible enough to allow non-deterministic LLM responses? (not too brittle)
3. Clear about what the answer should NOT contain?

Suggest an improved ground-truth description if needed. Keep it to 2-3 sentences."""

result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet',
        $${prompt}$$
    ) AS refined_ground_truth
""").collect()

print(f"Question: {sample_question['question']}")
print(f"\nOriginal ground truth: {sample_question['ground_truth']}")
print(f"\nRefinement suggestion:\n{result[0][0]}")

---
## Observability: Expected vs. Actual Question Distribution

Once your agent is deployed (even to a pilot), observability data reveals how actual usage
compares to your taxonomy assumptions. The cells below query mock data from `setup.sql`
to demonstrate the pattern.

In [ ]:
# Query the mock observability data to see actual question distribution
# In production, this queries SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY

session.sql("""
    SELECT
        QUESTION_CATEGORY,
        COUNT(*) AS question_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_total,
        ROUND(AVG(TOKENS_USED), 0) AS avg_tokens,
        ROUND(AVG(LATENCY_MS), 0) AS avg_latency_ms
    FROM MOCK_AGENT_USAGE
    GROUP BY QUESTION_CATEGORY
    ORDER BY question_count DESC
""").show()

In [ ]:
# Compare expected taxonomy distribution vs. actual

expected_distribution = {
    "lookup": 50,
    "aggregation": 25,
    "reasoning": 10,
    "policy": 10,
    "out_of_scope": 5,
}

actual = session.sql("""
    SELECT
        QUESTION_CATEGORY,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS actual_pct
    FROM MOCK_AGENT_USAGE
    GROUP BY QUESTION_CATEGORY
""").collect()

print(f"{'Category':<15} {'Expected %':>12} {'Actual %':>10} {'Delta':>8}")
print("-" * 50)
for row in actual:
    cat = row[0].lower()
    actual_pct = float(row[1])
    expected_pct = expected_distribution.get(cat, 0)
    delta = actual_pct - expected_pct
    flag = " <<<" if abs(delta) > 10 else ""
    print(f"{cat:<15} {expected_pct:>10.1f}% {actual_pct:>8.1f}% {delta:>+7.1f}%{flag}")

In [ ]:
# Find questions with high token usage (agent confusion signal)

session.sql("""
    SELECT
        USER_QUESTION,
        QUESTION_CATEGORY,
        TOKENS_USED,
        LATENCY_MS,
        SURFACE
    FROM MOCK_AGENT_USAGE
    WHERE TOKENS_USED > (
        SELECT AVG(TOKENS_USED) + 2 * STDDEV(TOKENS_USED)
        FROM MOCK_AGENT_USAGE
    )
    ORDER BY TOKENS_USED DESC
    LIMIT 10
""").show()

In [ ]:
# Surface and persona distribution

session.sql("""
    SELECT
        SURFACE,
        USER_ROLE,
        COUNT(*) AS interactions,
        ROUND(AVG(TOKENS_USED), 0) AS avg_tokens
    FROM MOCK_AGENT_USAGE
    GROUP BY SURFACE, USER_ROLE
    ORDER BY interactions DESC
""").show()

---
## Generate New Eval Questions from Production Gaps

Use high-token or unexpected-category questions from observability
to generate new eval dataset entries.

In [ ]:
# Generate eval entries for production gap questions

gap_questions = session.sql("""
    SELECT USER_QUESTION, QUESTION_CATEGORY
    FROM MOCK_AGENT_USAGE
    WHERE TOKENS_USED > (
        SELECT AVG(TOKENS_USED) + STDDEV(TOKENS_USED)
        FROM MOCK_AGENT_USAGE
    )
    ORDER BY TOKENS_USED DESC
    LIMIT 5
""").collect()

print("Generating eval entries for production gap questions...\n")
for row in gap_questions:
    question = row[0]
    category = row[1]
    
    prompt = f"""Generate a ground-truth description for evaluating an AI agent's response to this question.
The agent is a sales insights assistant with access to structured revenue/pipeline data.

Question: {question}
Category: {category}

Write a 2-3 sentence ground-truth description that:
- Describes what a correct answer should contain
- Notes what it should NOT contain or do
- Is specific enough to validate but flexible for non-deterministic responses"""
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-sonnet', $${prompt}$$) AS gt
    """).collect()
    
    print(f"Q: {question}")
    print(f"Category: {category}")
    print(f"Draft ground truth: {result[0][0][:200]}...")
    print()

---
## Agent Spec Document (Final Output)

Fill in the template below to produce your completed scoping document.
This synthesizes all workshop activities into a single deliverable.

### Agent Specification

| Field | Value |
|-------|-------|
| **Agent Name** | _[your_agent_name]_ |
| **Owner** | _[team and individual responsible]_ |
| **Personas Served** | _[from Activity 1]_ |
| **Question Categories** | _[from Activity 2]_ |
| **Tools** | _[from Activity 3 — selected tools for Phase 1]_ |
| **Out of Scope** | _[explicit refusal boundaries]_ |
| **Business Success Metric** | _[e.g., "80% of users get answers without filing a ticket"]_ |
| **Technical Metrics** | _[e.g., answer_correctness >= 0.75, tool_selection_accuracy >= 0.85]_ |
| **Eval Dataset** | _[pointer to SEED_EVAL_DATASET, target: 30+ questions]_ |
| **Phase 1 Scope** | _[persona + categories + tools that ship first]_ |
| **Phase 1 Exit Criteria** | _[metrics that must be met to proceed to Phase 2]_ |
| **Phase 2 Scope** | _[what expands: new tools, categories, personas]_ |
| **Known Blockers** | _[prerequisites not yet met from Activity 3]_ |

---
## Phased Delivery Plan

| Phase | Scope | Tools | Entry Criteria | Exit Criteria |
|-------|-------|-------|----------------|---------------|
| **1: Prove Value** | _[1 persona, lookup+aggregation]_ | _[Semantic View + Analyst]_ | Prerequisites met | answer_correctness >= 0.75, 5+ pilot users |
| **2: Expand Coverage** | _[add policy/reasoning]_ | _[+ Cortex Search]_ | Phase 1 exit met | tool_selection >= 0.85, correctness >= 0.80 |
| **3: Multi-Persona** | _[add Persona 2, routing/RBAC]_ | _[+ Agent Routing]_ | Phase 2 exit, Persona 2 taxonomy | per-persona evals pass |
| **4: Hardening** | _[budgets, CI/CD, SLAs]_ | _[+ monitoring]_ | Phase 3 exit, observability baseline | SLA met 30 consecutive days |

---
## Next Steps

1. **Formalize:** Transfer the spec above into your team's project tracking system
2. **Address blockers:** Build prerequisite artifacts (semantic view, search service) before the agent
3. **Grow eval dataset:** Expand from 10 seed questions to 30-50 covering all Phase 1 categories
4. **Build Phase 1 agent:** Use the spec as the requirements document
5. **Run baseline eval:** Evaluate the first version against your seed dataset
6. **Deploy to pilot:** Give 5-10 users access, observe usage patterns
7. **Iterate:** Use observability data to add production gaps to eval dataset

**Companion modules:**
- `evaluations/` — How to run and iterate on agent evaluations
- `cortex-ai-observability/` — Surface identification and cost attribution
- `agent_versioning/` — Safe iteration with named versions and aliases
- `agent-routing/` — Multi-agent orchestration for Phase 3+

In [ ]:
# Optional: Clean up lab resources (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS SCOPING_LAB").collect()
# session.sql("DROP WAREHOUSE IF EXISTS SCOPING_LAB_WH").collect()